In [ ]:
# showing kai how to github!

In [ ]:
import numpy as np
import pandas as pd

# extract info and data sections from original CSV

In [ ]:
def extract_info_and_data(parent_path, trial_name):
    import pandas as pd
    original_data_path = parent_path+ "/raw/" + trial_name

    # extract experiment info
    og_df = pd.read_csv(f"{original_data_path}.csv", skiprows=1)
    info_df = og_df[:8] # clip to only "info" section
    info_df = info_df.T # transpose so columns are titles, rows are data
    info_df.columns = info_df.iloc[0] # create new header row
    info_df = info_df[1:] # remove extra header row
    # save info df
    info_df.to_csv(parent_path+"/extracted/"+trial_name+"_info.csv", index = False)

    # create and save data df
    data_df = pd.read_csv(f"{original_data_path}.csv", header = 9) # remove info rows, keep only data rows with Time, and probes as headers

    # rename Probe A and B to quadrature and pre-mixed wave
    data_df.columns = ['Time (s)', 'Quadrature (V)', "Pre-Mixed Wave (V)"]

    # save new data df
    data_df.to_csv(parent_path+"/extracted/"+trial_name+"_data.csv", index = False)
    print(parent_path+"/extracted/"+trial_name+"_data.csv")

    # print new csv file names

    filepaths = [parent_path+"/extracted/"+trial_name+"_info.csv", parent_path+"/extracted/"+trial_name+"_data.csv"]

    # print("experiment INFO saved to: \n", parent_path+trial_name+"_info.csv \n")
    # print("experiment DATA saved to: \n", parent_path+trial_name+"_data.csv")

    return filepaths


# putting together multiple trial runs

In [ ]:

def combine_trials(loc, extension, variable, block, level = None):
    # extract trials for desired variable from data folder
    import os
    import pandas as pd

    if level is not None:
        folderpath = loc + level
    else:
        folderpath = loc

    print(folderpath)
    files = [
        filename for filename in os.listdir(folderpath)
        if filename.endswith(extension) and variable in filename
    ]
    print('made files list')
    files = [os.path.splitext(f)[0] for f in files]
    print(files)

    # extract data contents from files
    trial_data_paths = []
    for f in files:
        trial_n_contents_path = extract_info_and_data(loc, f)
        trial_data_paths.append(trial_n_contents_path[1])

    # empty data frames for creating combined df, averages at each time of the three trials
    combined_df = pd.DataFrame()

    trial = pd.read_csv(trial_data_paths[0])
    combined_df["Time (s)"] = trial["Time (s)"]
    col_names_q = []
    col_names_pw = []
    
    for t in list(range(0, len(trial_data_paths))):
        trial_df = pd.read_csv(trial_data_paths[t])
        quad_col = f"Quad_V_Trial{t+1}"
        col_names_q.append(quad_col)
        combined_df[quad_col] = trial_df['Quadrature (V)']
        premix_col = f"PreMix_V_Trial{t+1}"
        col_names_pw.append(premix_col)
        combined_df[premix_col] = trial_df['Pre-Mixed Wave (V)']

    combined_df["Avg. Quadrature (V)"] = combined_df[col_names_q].mean(axis=1)
    combined_df["Avg. Pre-Mixed Wave (V)"] = combined_df[col_names_pw].mean(axis=1)
    column_names = list(combined_df.columns)
    new_order = [column_names[0]]+column_names[-2:] + column_names[1:-2]
    combined_df = combined_df[new_order]

    combined_df.to_csv( loc+"/combined/"+f"{block}{variable}combinedtrials.csv")
    print("New combined trials dataframe saved to", loc+"/combined/"+f"{block}{variable}combinedtrials.csv")

    return combined_df

# run fxn on trial data
location = "/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1"
combo = combine_trials(location, ".csv", "_theta_", "a1", "/raw")

/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1/raw
made files list
['calib_a1_50khz_trial1_theta_20260702_115619', 'calib_a1_50khz_trial2_theta_20260702_115937', 'calib_a1_50khz_trial3_theta_20260702_120237']
/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1/extracted/calib_a1_50khz_trial1_theta_20260702_115619_data.csv
/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1/extracted/calib_a1_50khz_trial2_theta_20260702_115937_data.csv
/Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1/extracted/calib_a1_50khz_trial3_theta_20260702_120237_data.csv
New combined trials dataframe saved to /Users/mauriellenoto/Desktop/seager/uvphasedelay/data/a1/combined/a1_theta_combinedtrials.csv
